# Segmentation d'images avec PyTorch Lightning

In [ ]:
!pip install -q lightning torchmetrics timm tifffile

In [ ]:
import random

import lightning
import matplotlib
import matplotlib.pyplot as plt
import numpy
import pandas
import torch
import torchmetrics
from lightning.pytorch.callbacks import EarlyStopping, ModelCheckpoint
from lightning.pytorch.loggers import CSVLogger
from lightning.pytorch.utilities.model_summary import ModelSummary
from tifffile import imread
from torch import nn
from torch.utils.data import DataLoader, Dataset
from torchvision.transforms.v2 import functional as transforms

In [ ]:
!nvidia-smi

In [ ]:
%%time
!rm -rf sample_data
!git clone https://github.com/mlambda/ign-FNF-2009.git

## Chargement des données

Les données sont constituées de 20 grandes tuiles cartographiques de l'IGN. Chaque grande tuile est elle-même un carré de 3 × 3 tuiles de 1024 × 1024 pixels.

2 tuiles ont été mises de côté pour la validation, et 2 autres tuiles pour le test.

In [ ]:
df_train = pandas.read_csv("ign-FNF-2009/train-files.csv")
df_val = pandas.read_csv("ign-FNF-2009/val-files.csv")
df_test = pandas.read_csv("ign-FNF-2009/test-files.csv")

print("Coordonnées en entraînement")
print(df_train.coord.unique())

print("Coordonnées en val")
print(df_val.coord.unique())

print("Coordonnées en test")
print(df_test.coord.unique())

df_test

## Étude d'un exemple et de son annotation

In [ ]:
def read_observation(path: str) -> torch.Tensor:
  """Read a tile as a (channels, height, width) tensor."""
  return torch.from_numpy(imread(path)).permute(2, 0, 1)


def read_annotation(path: str) -> torch.Tensor:
  """Read a one-hot annotation as a (1, height, width) mask."""
  return torch.from_numpy(imread(path)).argmax(dim=-1)[None].float()


example_input = read_observation(df_train.Observations[0])
example_annotation = read_annotation(df_train.Annotations[0])

In [ ]:
# Hack nécessaire pour que imshow affiche correctement les tuiles avec seulement
# de la forêt (sinon les 1 représentant les pixels de forêt sont remplacés par
# des 0 si la tuile est consitutée seulement de 1s).
class NoopNormalize(matplotlib.colors.Normalize):
  def __init__(self, vmin=None, vmax=None, vcenter=None, clip=False):
    super().__init__(vmin, vmax, clip)
    self.vcenter = vcenter

  def __call__(self, value, clip=None):
    return value


_noop_normalize = NoopNormalize()


def to_displayable(image: torch.Tensor) -> numpy.ndarray:
  """Turn a (channels, height, width) tensor into an array imshow can plot."""
  image = image.detach().cpu()
  if image.ndim == 2:
    return image.numpy()
  if image.shape[0] == 3:
    return image.permute(1, 2, 0).numpy()
  if image.shape[0] == 2:
    return image[1].numpy()
  return image[0].numpy()


def display_sample(*images: torch.Tensor) -> None:
  """Show side-by-side an input image, the ground truth and the prediction."""
  figure, axes = plt.subplots(1, len(images), figsize=(18, 18))

  titles = ["Entrée", "Cible", "Prédiction"]
  for image, ax, title in zip(images, axes, titles):
    ax.set_title(title)
    ax.imshow(to_displayable(image), norm=_noop_normalize)
    ax.axis("off")
  plt.show()


display_sample(example_input, example_annotation)

## Options générales

In [ ]:
# Dimension des images originales
dim_original = 1024

# Dimension des images en entrée du réseau
dim_input = 512

# Nombre de canaux des images en entrée
num_channel = 3

# Nombre de processus utilisés pour charger les données
num_workers = 2

## Augmentation des données

En traitement d'images, il est extrêmement important d'augmenter les données par des transformations pour rendre le réseau appris moins sensibles aux rotations, recadrages, etc. Implémentez la fonction `transform_data`, qui, si `eval` vaut `True` se contentera de faire un recadrage centré des images `X` et `Y`, et sinon fera toutes les transformations que vous jugerez opportunes sur ces mêmes tenseurs pour l'entraînement (recadrage aléatoire, miroir, rotation, bruitage des couleurs, de la luminosité, etc).

Pour ce faire, utilisez les fonctions du module [`torchvision.transforms.v2.functional`](https://docs.pytorch.org/vision/stable/transforms.html#functional-transforms), importé ici sous le nom `transforms`.

In [ ]:
def transform_data(
  X: torch.Tensor,
  Y: torch.Tensor,
  eval: bool = False,
  cropped_size: tuple[int, int] = (dim_input, dim_input),
) -> tuple[torch.Tensor, torch.Tensor]:
  # Votre code ici
  return X, Y


# display_sample(*transform_data(example_input, example_annotation))

### Solution

In [ ]:
def transform_data(
  X: torch.Tensor,
  Y: torch.Tensor,
  eval: bool = False,
  cropped_size: tuple[int, int] = (dim_input, dim_input),
) -> tuple[torch.Tensor, torch.Tensor]:
  height, width = X.shape[-2:]
  cropped_height, cropped_width = cropped_size
  if eval:
    # Recadrage centré
    crop_params = dict(
      top=(height - cropped_height) // 2,
      left=(width - cropped_width) // 2,
      height=cropped_height,
      width=cropped_width,
    )
    X = transforms.crop(X, **crop_params)
    Y = transforms.crop(Y, **crop_params)
  else:
    # Rotation aléatoire
    k = random.choice([None, 1, 3])
    if k is not None:
      X = torch.rot90(X, k, dims=(-2, -1))
      Y = torch.rot90(Y, k, dims=(-2, -1))

    # Recadrage aléatoire
    crop_params = dict(
      top=random.randint(0, height - cropped_height),
      left=random.randint(0, width - cropped_width),
      height=cropped_height,
      width=cropped_width,
    )
    X = transforms.crop(X, **crop_params)
    Y = transforms.crop(Y, **crop_params)

    # Miroir horizontal
    if random.random() > 0.5:
      X = transforms.horizontal_flip(X)
      Y = transforms.horizontal_flip(Y)

    # Miroir vertical
    if random.random() > 0.5:
      X = transforms.vertical_flip(X)
      Y = transforms.vertical_flip(Y)
  return X, Y


for _ in range(10):
  display_sample(*transform_data(example_input, example_annotation))

## Création d'un `Dataset` PyTorch

Pour décrire nos données, nous allons utiliser la classe [`torch.utils.data.Dataset`](https://docs.pytorch.org/docs/stable/data.html#map-style-datasets). Il faut principalement deux méthodes : `__getitem__`, qui est appelée pour récupérer **un** exemple à partir de son indice, et `__len__`, qui est appelée pour connaître le nombre total d'exemples. C'est ensuite le [`DataLoader`](https://docs.pytorch.org/docs/stable/data.html#torch.utils.data.DataLoader) qui regroupe ces exemples en batchs, mélange le dataset avant chaque epoch et parallélise le chargement.

In [ ]:
class SegmentationDataset(Dataset):
  """Load samples with appropriate transforms."""

  def __init__(
    self,
    df: pandas.DataFrame,
    eval: bool,
    cropped_size: tuple[int, int] = (dim_input, dim_input),
  ) -> None:
    """Initialize the dataset."""
    # Votre code ici

  def __len__(self):
    """Compute the number of samples."""
    # Votre code ici

  def __getitem__(self, index):
    """Generate one sample of data."""
    # Votre code ici

### Solution

In [ ]:
class SegmentationDataset(Dataset):
  """Load samples for PyTorch."""

  def __init__(
    self,
    df: pandas.DataFrame,
    eval: bool,
    cropped_size: tuple[int, int] = (dim_input, dim_input),
  ) -> None:
    """Initialize the dataset."""
    self.X_paths = df.Observations.to_numpy()
    self.Y_paths = df.Annotations.to_numpy()
    assert self.X_paths.shape[0] == self.Y_paths.shape[0]
    self.eval = eval
    self.cropped_size = cropped_size

  def __len__(self) -> int:
    """Compute the number of samples."""
    return self.X_paths.shape[0]

  def __getitem__(self, index: int) -> tuple[torch.Tensor, torch.Tensor]:
    """Generate one sample of data."""
    X, Y = transform_data(
      read_observation(self.X_paths[index]),
      read_annotation(self.Y_paths[index]),
      self.eval,
      self.cropped_size,
    )
    return X.float() / 255.0, Y


def make_loader(df: pandas.DataFrame, batch_size: int, eval: bool) -> DataLoader:
  """Wrap a dataset in a data loader: batching, shuffling and prefetching."""
  return DataLoader(
    SegmentationDataset(df, eval),
    batch_size=batch_size,
    shuffle=not eval,
    num_workers=num_workers,
    persistent_workers=num_workers > 0,
  )


loader = make_loader(df_train, 3, eval=True)
X, Y = next(iter(loader))
X.shape, Y.shape

## Création du modèle

Pour le modèle, nous devons passer d'une image de dimension (batch × 3 × hauteur × largeur) à une sortie de dimension (batch × 1 × hauteur × largeur).

Vous pouvez vous inspirer de l'architecture vue en démonstration des autoencodeurs débruiteurs. Cette fois cependant, on ne réduira pas les capacités du réseau au centre du modèle.

Vous pouvez aussi consulter [la description de l'architecture U-net](https://paperswithcode.com/method/u-net) pour plus d'insipration.

In [ ]:
def get_model(starting_depth: int = 32, steps: int = 5) -> nn.Module:
  pass  # Votre code ici

### Solution

In [ ]:
# Implémentation du modèle U-net tel que décrit sur la page
# https://paperswithcode.com/method/u-net
# Vous pouvez aussi implémenter tout autre modèle adapté :
# https://paperswithcode.com/methods/category/segmentation-models


def downsample() -> nn.MaxPool2d:
  return nn.MaxPool2d(kernel_size=2)


def upsample() -> nn.Upsample:
  return nn.Upsample(scale_factor=2)


def semiblock(in_channels: int, feature_maps: int) -> nn.Sequential:
  return nn.Sequential(
    nn.Conv2d(in_channels, feature_maps, 3, padding="same"),
    nn.BatchNorm2d(feature_maps),
    nn.ReLU(),
    nn.Dropout(0.1),
  )


def block(in_channels: int, feature_maps: int) -> nn.Sequential:
  return nn.Sequential(
    semiblock(in_channels, feature_maps), semiblock(feature_maps, feature_maps)
  )


class UNet(nn.Module):
  """U-net: an encoder, a decoder, and skip connections between the two."""

  def __init__(self, starting_depth: int = 32, steps: int = 5) -> None:
    super().__init__()
    depths = [min(512, starting_depth * 2**i) for i in range(steps)]

    self.downsample = downsample()
    self.upsample = upsample()

    # Encodeur : un bloc par niveau, la résolution étant divisée par deux à
    # chaque niveau
    self.encoder = nn.ModuleList(
      [
        block(depths[i - 1] if i else num_channel, depth)
        for i, depth in enumerate(depths)
      ]
    )

    # Décodeur : à chaque niveau, l'encodage de même résolution est concaténé au
    # tenseur sur-échantillonné, d'où le nombre de canaux en entrée
    self.decoder = nn.ModuleList(
      [
        block(depth + previous_depth, depth)
        for depth, previous_depth in zip(depths[-2::-1], depths[::-1])
      ]
    )

    self.head = nn.Conv2d(depths[0], 1, 1)

  def forward(self, inputs: torch.Tensor) -> torch.Tensor:
    encodings = []
    for i, encoder_block in enumerate(self.encoder):
      encodings.append(encoder_block(self.downsample(encodings[-1]) if i else inputs))

    x = encodings[-1]
    for decoder_block, encoding in zip(self.decoder, encodings[-2::-1]):
      x = decoder_block(torch.cat([encoding, self.upsample(x)], dim=1))

    return self.head(x)


def get_model(starting_depth: int = 32, steps: int = 5) -> nn.Module:
  return UNet(starting_depth, steps)

## Entraînement

On peut maintenant utiliser notre `Dataset` pour entraîner notre modèle. Le `LightningModule` ci-dessous décrit la fonction de perte, les métriques et l'optimiseur : le modèle renvoie des logits, la sigmoïde étant appliquée par la fonction de perte pendant l'apprentissage et explicitement au moment de la prédiction.

In [ ]:
class ImageSegmenter(lightning.LightningModule):
  """Lightning wrapper for binary segmentation models."""

  def __init__(self, model: nn.Module, learning_rate: float = 1e-4) -> None:
    super().__init__()
    self.save_hyperparameters(ignore=["model"])
    self.model = model
    # Une entrée d'exemple permet à Lightning d'afficher la forme des tenseurs
    # d'entrée et de sortie de chaque couche dans le résumé du modèle
    self.example_input_array = torch.zeros(1, num_channel, dim_input, dim_input)
    # torchmetrics demande une instance de métrique par étape
    self.accuracies = nn.ModuleDict(
      {
        f"{stage}_accuracy": torchmetrics.Accuracy(task="binary")
        for stage in ("train", "val", "test")
      }
    )

  def forward(self, images: torch.Tensor) -> torch.Tensor:
    return self.model(images)

  def _step(self, batch: tuple[torch.Tensor, torch.Tensor], stage: str) -> torch.Tensor:
    images, annotations = batch
    logits = self(images)
    loss = nn.functional.binary_cross_entropy_with_logits(logits, annotations)
    accuracy = self.accuracies[f"{stage}_accuracy"]
    accuracy(logits.sigmoid(), annotations.int())
    self.log(f"{stage}_loss", loss, on_step=False, on_epoch=True, prog_bar=True)
    self.log(f"{stage}_accuracy", accuracy, on_step=False, on_epoch=True, prog_bar=True)
    return loss

  def training_step(
    self, batch: tuple[torch.Tensor, torch.Tensor], batch_index: int
  ) -> torch.Tensor:
    return self._step(batch, "train")

  def validation_step(
    self, batch: tuple[torch.Tensor, torch.Tensor], batch_index: int
  ) -> torch.Tensor:
    return self._step(batch, "val")

  def test_step(
    self, batch: tuple[torch.Tensor, torch.Tensor], batch_index: int
  ) -> torch.Tensor:
    return self._step(batch, "test")

  def predict_step(
    self, batch: torch.Tensor | tuple[torch.Tensor, ...], batch_index: int
  ) -> torch.Tensor:
    images = batch[0] if isinstance(batch, (list, tuple)) else batch
    return self(images).sigmoid().cpu()

  def configure_optimizers(self) -> torch.optim.Optimizer:
    return torch.optim.Adam(self.parameters(), lr=self.hparams.learning_rate)

In [ ]:
# Votre code ici

### Solution

In [ ]:
# Construction du modèle
segmenter = ImageSegmenter(get_model(), learning_rate=1e-4)
print(ModelSummary(segmenter, max_depth=-1))

batch_size = 4

# Création des chargeurs de données
train_loader = make_loader(df_train, batch_size, eval=False)
val_loader = make_loader(df_val, batch_size, eval=True)

# Entraînement du modèle, en évaluation sur le loader de validation à la fin de
# chaque epoch
epochs = 15
trainer = lightning.Trainer(
  max_epochs=epochs,
  accelerator="auto",
  devices=1,
  logger=CSVLogger("logs", name="u_net"),
  callbacks=[
    EarlyStopping(monitor="val_loss", patience=5),
    ModelCheckpoint(
      monitor="val_accuracy", mode="max", save_weights_only=True, save_top_k=1
    ),
  ],
)
trainer.fit(segmenter, train_loader, val_loader)

## Application aux données de test

Une fois votre modèle suffisamment appris, appliquez-le aux données de test : les prédictions sont-elles de bonne qualité ?

In [ ]:
# Votre code ici

### Solution

In [ ]:
test_loader = make_loader(df_test, 8, eval=True)

# `ckpt_path="best"` recharge les poids sauvegardés par le ModelCheckpoint
trainer.test(segmenter, dataloaders=test_loader, ckpt_path="best")

# `predict` renvoie une liste de tenseurs, un par batch : on les concatène
preds = torch.cat(trainer.predict(segmenter, test_loader))

print(len(test_loader.dataset), preds.shape)

for i, p in enumerate(preds[:8]):
  X, Y = test_loader.dataset[i]
  display_sample(X, Y, p)

## Modèle pré-entraîné

In [ ]:
import timm


class Normalize(nn.Module):
  """Normalize images already scaled to [0, 1], channel by channel."""

  def __init__(
    self, mean: tuple[float, ...] | float, std: tuple[float, ...] | float
  ) -> None:
    super().__init__()
    self.register_buffer("mean", torch.tensor(mean).reshape(1, -1, 1, 1))
    self.register_buffer("std", torch.tensor(std).reshape(1, -1, 1, 1))

  def forward(self, images: torch.Tensor) -> torch.Tensor:
    return (images - self.mean) / self.std


class Frozen(nn.Module):
  """Wrap a module whose weights are frozen and kept in evaluation mode."""

  def __init__(self, module: nn.Module) -> None:
    super().__init__()
    self.module = module.requires_grad_(False)
    self.eval()

  def train(self, mode: bool = True) -> "Frozen":
    return super().train(False)

  def forward(self, images: torch.Tensor) -> torch.Tensor:
    return self.module(images)


base_model = timm.create_model(
  "tf_efficientnet_b7.aa_in1k", pretrained=True, num_classes=0, global_pool=""
)
data_config = timm.data.resolve_model_data_config(base_model)

model = nn.Sequential(
  Normalize(data_config["mean"], data_config["std"]),
  # (2560, 16, 16)
  Frozen(base_model),
  # Votre code
)

segmenter = ImageSegmenter(model, learning_rate=1e-4)

print(ModelSummary(segmenter, max_depth=2))

# trainer = lightning.Trainer(
#     max_epochs=epochs,
#     accelerator="auto",
#     devices=1,
#     logger=CSVLogger("logs", name="efficientnet_b7"),
#     callbacks=[EarlyStopping(monitor="val_loss", patience=5),
#                ModelCheckpoint(monitor="val_accuracy",
#                                mode="max",
#                                save_weights_only=True,
#                                save_top_k=1)])
# trainer.fit(segmenter, train_loader, val_loader)

### Solution

In [ ]:
base_model = timm.create_model(
  "tf_efficientnet_b7.aa_in1k", pretrained=True, num_classes=0, global_pool=""
)
data_config = timm.data.resolve_model_data_config(base_model)

starting_depth = 512


def conv(in_channels: int, out_channels: int, kernel_size: int = 3) -> nn.Sequential:
  return nn.Sequential(
    nn.Conv2d(in_channels, out_channels, kernel_size, padding="same"), nn.ReLU()
  )


model = nn.Sequential(
  Normalize(data_config["mean"], data_config["std"]),
  # (2560, 16, 16)
  Frozen(base_model),
  # (512, 16, 16)
  conv(base_model.num_features, starting_depth, kernel_size=1),
  # (512, 32, 32)
  nn.Upsample(scale_factor=2),
  # (256, 32, 32)
  conv(starting_depth, starting_depth // 2),
  # (256, 64, 64)
  nn.Upsample(scale_factor=2),
  # (128, 64, 64)
  conv(starting_depth // 2, starting_depth // 4),
  # (128, 128, 128)
  nn.Upsample(scale_factor=2),
  # (64, 128, 128)
  conv(starting_depth // 4, starting_depth // 8),
  # (64, 256, 256)
  nn.Upsample(scale_factor=2),
  # (32, 256, 256)
  conv(starting_depth // 8, starting_depth // 16),
  # (32, 512, 512)
  nn.Upsample(scale_factor=2),
  # (16, 512, 512)
  conv(starting_depth // 16, starting_depth // 32),
  nn.Dropout(0.9),
  # (1, 512, 512)
  nn.Conv2d(starting_depth // 32, 1, 3, padding="same"),
)

segmenter = ImageSegmenter(model, learning_rate=1e-4)

print(ModelSummary(segmenter, max_depth=2))

trainer = lightning.Trainer(
  max_epochs=epochs,
  accelerator="auto",
  devices=1,
  logger=CSVLogger("logs", name="efficientnet_b7"),
  callbacks=[
    EarlyStopping(monitor="val_loss", patience=5),
    ModelCheckpoint(
      monitor="val_accuracy", mode="max", save_weights_only=True, save_top_k=1
    ),
  ],
)
trainer.fit(segmenter, train_loader, val_loader)

In [ ]:
test_loader = make_loader(df_test, 8, eval=True)

# `ckpt_path="best"` recharge les poids sauvegardés par le ModelCheckpoint
trainer.test(segmenter, dataloaders=test_loader, ckpt_path="best")

# `predict` renvoie une liste de tenseurs, un par batch : on les concatène
preds = torch.cat(trainer.predict(segmenter, test_loader))

print(len(test_loader.dataset), preds.shape)

for i, p in enumerate(preds[:8]):
  X, Y = test_loader.dataset[i]
  display_sample(X, Y, p)